In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

EPOCH_1_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_1_TOKENIZED"], "buckets_split_validate"),
}

EPOCH_2_FOLDERS = {
    "train": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_train"),
    "validate": os.path.join(os.environ["GPTNL_2_TOKENIZED"], "buckets_split_validate"),
}

In [2]:
import random


def get_bucket_from_fp(file_path):
    if "code" in file_path:
        return "code"
    elif "dutch_high_quality" in file_path:
        return "dutch_high_quality"
    elif "dutch_low_medium_quality" in file_path:
        return "dutch_low_medium_quality"
    elif "english_high_quality" in file_path:
        return "english_high_quality"
    elif "english_low_medium_quality" in file_path:
        return "english_low_medium_quality"
    elif "new_high_quality" in file_path:
        return "new_high_quality"
    elif "new_low_medium_quality" in file_path:
        return "new_low_medium_quality"
    elif "other_languages" in file_path:
        return "other_languages"
    else:
        raise ValueError(f"Unknown bucket for file path: {file_path}")


DEBUG_FOLDER = os.environ("GPTNL_DEBUG")

buckets = [
    "code",
    "dutch_high_quality",
    "dutch_low_medium_quality",
    "english_high_quality",
    "english_low_medium_quality",
    "new_high_quality",  # only exists in epoch 2
    "new_low_medium_quality",  # only exists in epoch 2
    "other_languages",
]

file_per_bucket_epoch_1 = {k: [] for k in buckets}
file_per_bucket_epoch_2 = {k: [] for k in buckets}

del file_per_bucket_epoch_1["new_high_quality"]
del file_per_bucket_epoch_1["new_low_medium_quality"]

# os.listdir(EPOCH_1_FOLDERS["train"])
# os.listdir(EPOCH_2_FOLDERS["train"])

files_list_epoch_1 = os.listdir(EPOCH_1_FOLDERS["train"])
random.shuffle(files_list_epoch_1)

for file in files_list_epoch_1:
    bucket = get_bucket_from_fp(file)
    file_path = os.path.join(EPOCH_1_FOLDERS["train"], file)

    if len(file_per_bucket_epoch_1[bucket]) < 2:
        file_per_bucket_epoch_1[bucket].append(file_path)

    any_left = False
    for k, v in file_per_bucket_epoch_1.items():
        if v is None or len(v) < 2:
            any_left = True
            break

    if any_left:
        continue
    else:
        print("Done! for epoch 1")
        break


for bucket_subdir in os.listdir(EPOCH_2_FOLDERS["train"]):
    bucket = get_bucket_from_fp(bucket_subdir)
    bucket_subdir_fp = os.path.join(EPOCH_2_FOLDERS["train"], bucket_subdir)

    files_list = os.listdir(bucket_subdir_fp)
    random.shuffle(files_list)

    for file in files_list:
        file_path = os.path.join(bucket_subdir_fp, file)

        if len(file_per_bucket_epoch_2[bucket]) < 2:
            file_per_bucket_epoch_2[bucket].append(file_path)

        any_left = False
        for k, v in file_per_bucket_epoch_2.items():
            if v is None or len(v) < 2:
                any_left = True
                break

        if any_left:
            continue
        else:
            print("Done! for epoch 2")
            break

Done! for epoch 1
Done! for epoch 2


In [3]:
file_per_bucket_epoch_1

{'code': ['GPTNL_1_TOKENIZED/buckets_split_train/code_1748955070_102_793045_rows_train_tokenized.npy',
  'GPTNL_1_TOKENIZED/buckets_split_train/code_1748955070_103_781000_rows_train_tokenized.npy'],
 'dutch_high_quality': ['GPTNL_1_TOKENIZED/buckets_split_train/dutch_high_quality',
  'GPTNL_1_TOKENIZED/buckets_split_train/dutch_high_quality_1748945813_2_87667_rows_train_tokenized.npy'],
 'dutch_low_medium_quality': ['GPTNL_1_TOKENIZED/buckets_split_train/dutch_low_medium_quality_1748946275_4_311199_rows_train_tokenized.npy',
  'GPTNL_1_TOKENIZED/buckets_split_train/dutch_low_medium_quality_1748946275_11_566647_rows_train_tokenized.npy'],
 'english_high_quality': ['GPTNL_1_TOKENIZED/buckets_split_train/english_high_quality_1748946595_21_34567_rows_train_tokenized.npy',
  'GPTNL_1_TOKENIZED/buckets_split_train/english_high_quality_1748946595_52_34578_rows_train_tokenized.npy'],
 'english_low_medium_quality': ['GPTNL_1_TOKENIZED/buckets_split_train/english_low_medium_quality_1748947879_47

In [4]:
file_per_bucket_epoch_2

{'code': ['GPTNL_2_TOKENIZED/buckets_split_train/code/code_1754151838_21_676000_rows_train_tokenized.npy',
  'GPTNL_2_TOKENIZED/buckets_split_train/code/code_1754148971_43_677000_rows_train_tokenized.npy'],
 'dutch_high_quality': ['GPTNL_2_TOKENIZED/buckets_split_train/dutch_high_quality/dutch_high_quality_1754139956_43_292185_rows_train_tokenized.npy',
  'GPTNL_2_TOKENIZED/buckets_split_train/dutch_high_quality/dutch_high_quality_1754139956_28_88784_rows_train_tokenized.npy'],
 'dutch_low_medium_quality': ['GPTNL_2_TOKENIZED/buckets_split_train/dutch_low_medium_quality/dutch_low_medium_quality_1754140620_7_364770_rows_train_tokenized.npy',
  'GPTNL_2_TOKENIZED/buckets_split_train/dutch_low_medium_quality/dutch_low_medium_quality_1754140620_8_330859_rows_train_tokenized.npy'],
 'english_high_quality': ['GPTNL_2_TOKENIZED/buckets_split_train/english_high_quality/english_high_quality_1754140878_34_34559_rows_train_tokenized.npy',
  'GPTNL_2_TOKENIZED/buckets_split_train/english_high_qual

In [5]:
import numpy as np


def count_tokens(file_path: str, dtype=np.uint32):
    array = np.memmap(file_path, dtype=dtype)
    len = array.shape[0]
    return len


for epoch, files_per_bucket in zip(
    ["epoch_1", "epoch_2"], [file_per_bucket_epoch_1, file_per_bucket_epoch_2]
):
    for bucket, filepaths in files_per_bucket.items():
        for filepath in filepaths:
            token_count = count_tokens(filepath)
            print(f"Epoch {epoch} Bucket: {bucket}, Tokens: {token_count / 10e6:.2f}M")

            new_bucket_folder = os.path.join(DEBUG_FOLDER, f"{epoch}_{bucket}")
            # if os.path.exists(new_bucket_folder):
            # print(f"Creating folder: {new_bucket_folder}")
            # shutil.rmtree(new_bucket_folder)

            os.makedirs(new_bucket_folder, exist_ok=True)

            filename = os.path.basename(filepath)
            symlink_path = os.path.join(new_bucket_folder, filename)
            if os.path.exists(symlink_path):
                # print(f"Removing existing symlink: {symlink_path}")
                os.remove(symlink_path)

            os.symlink(filepath, symlink_path)

    print()

Epoch epoch_1 Bucket: code, Tokens: 55.68M
Epoch epoch_1 Bucket: code, Tokens: 55.36M
